# Searching NIH pub med DB for abstracts
API Ref and examples: https://www.ncbi.nlm.nih.gov/books/NBK25500/

In [10]:
import requests
import pandas as pd
import re 
api_search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
api_fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

In [2]:
def search_nih_api(search_string):
    params = {
        "db": "pubmed",
        "term": search_string,
        "retmax": 100,
        "retmode": "json"
    }
    publication_ids = requests.get(api_search_url, params=params).json()
    id_list = publication_ids["esearchresult"]["idlist"]
    return id_list

def get_abstracts_from_id_list(id_list):
    abstracts = requests.get(api_fetch_url, params={
        "db": "pubmed",
        "id": ",".join(id_list),
        "rettype": "abstract",
        "retmode": "text"
    })
    #only need the text
    return abstracts.text

In [7]:
search_terms = ["colorectal cancer prognosis", 
                "ucec cancer prognosis", 
                "pancreatic cancer prognosis",
               "breast cancer prognosis"]


for term in search_terms:
    article_ids = search_nih_api(term)
    results_text = get_abstracts_from_id_list(article_ids)
    #write out text
    with open(f'{term}.txt', "w") as outfile:
        outfile.write(results_text)
    

In [6]:
#print out the a full GET request from the search terms as formatted for an API call, and the response

api_string = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pubmed&term=pancreatic+cancer"
response = requests.get(api_string)
print(response.url)
print(response.text)


https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pubmed&term=pancreatic+cancer
<?xml version="1.0" encoding="UTF-8" ?>
<!DOCTYPE eSearchResult PUBLIC "-//NLM//DTD esearch 20060628//EN" "https://eutils.ncbi.nlm.nih.gov/eutils/dtd/20060628/esearch.dtd">
<eSearchResult><Count>164766</Count><RetMax>20</RetMax><RetStart>0</RetStart><IdList>
<Id>42299576</Id>
<Id>42299458</Id>
<Id>42299399</Id>
<Id>42299111</Id>
<Id>42298970</Id>
<Id>42298760</Id>
<Id>42298691</Id>
<Id>42298507</Id>
<Id>42298141</Id>
<Id>42298081</Id>
<Id>42298078</Id>
<Id>42298062</Id>
<Id>42297911</Id>
<Id>42297776</Id>
<Id>42297743</Id>
<Id>42297425</Id>
<Id>42297317</Id>
<Id>42296646</Id>
<Id>42296390</Id>
<Id>42296185</Id>
</IdList><TranslationSet><Translation>     <From>pancreatic cancer</From>     <To>"pancreatic neoplasms"[MeSH Terms] OR ("pancreatic"[All Fields] AND "neoplasms"[All Fields]) OR "pancreatic neoplasms"[All Fields] OR ("pancreatic"[All Fields] AND "cancer"[All Fields]) OR "pancreatic cancer

In [ ]:
#read the pancreatic abstracts back in
with open("pancreatic cancer prognosis.txt") as f:
    pancreatic_abstracts = f.read()


#parse 1st abstract
first_abstract = pancreatic_abstracts.split("\n\n\n")[0]

title = first_abstract.split("\n\n")[1]

#parse out background and results. these are indicated by a "BACKGROUND" and "RESULTS" heading
background_match = re.search(r"BACKGROUND:(.*?)RESULTS:", first_abstract, re.DOTALL)
results_match = re.search(r"RESULTS:(.*?)(?:CONCLUSIONS:|$)", first_abstract, re.DOTALL)
background = background_match.group(1).strip() if background_match else ""
results = results_match.group(1).strip() if results_match else ""
print("Title:", title)
print('\n\n')
print("Background:", background)
print('\n\n')
print("Results:", results)

Title: Distinct epidemiology and treatment outcomes between skeletal and extraskeletal 
Ewing sarcoma in Japan: a population-based study.



Background: Ewing sarcoma arises at both skeletal (SES) and extraskeletal (EES) 
sites; however, whether the anatomical origin influences outcomes in current 
practice remains uncertain. This study compared clinical characteristics, 
treatment, and survival rates of patients with SES and EES in a large nationwide 
Japanese cohort.
METHODS: We analyzed patients diagnosed with Ewing's sarcoma between 2016 and 
2019 in Japan, classified as having SES or EES, using a population-based cancer 
registry. Demographics, stage, treatment, hospital characteristics, and overall 
survival (OAS) were evaluated using chi-square tests, Kaplan-Meier estimates, 
and log-rank analyses.



Results: We identified 505 patients with ES: 211 with SES and 294 with EES. 
Patients with EES were significantly older than those with SES (P < .001). 
Chemotherapy (97.7% vs. 74.